# 00 Colab 課前環境與資料檢查

這份 notebook 給學生在正式上課前執行。目的不是先學完量化交易，而是確認 Colab、Python 套件、資料讀取、簡單圖表與 CSV 輸出都可以正常運作。

**課前成功標準**

- 可以在 Colab 開啟並執行所有 cells。
- 可以成功建立一份 OHLCV 資料表。
- 可以產生 features 與 target。
- 可以儲存 `market_data.csv` 與 `features.csv`。
- 知道本課程不提供投資建議，回測不代表未來績效。

**建議做法**：在 Colab 中選擇 `Runtime > Run all`。若有錯誤，截圖錯誤訊息帶到第一堂課。

## 1. 檢查 Python 與常用套件

本課程主要使用 `pandas`、`numpy`、`matplotlib`、`scikit-learn`。`yfinance` 是選用套件；若下載市場資料失敗，課堂 notebook 會自動改用合成資料。

In [ ]:
import sys, platform, importlib.util

print('Python:', sys.version)
print('Platform:', platform.platform())

required = ['numpy', 'pandas', 'matplotlib', 'sklearn']
optional = ['yfinance']

for pkg in required + optional:
    status = 'OK' if importlib.util.find_spec(pkg) else 'MISSING'
    label = 'required' if pkg in required else 'optional'
    print(f'{pkg:12s} {label:8s} {status}')

## 2. Colab 選用：安裝 yfinance

如果上一格顯示 `yfinance MISSING`，而你希望下載真實市場資料，可以把下面的 `INSTALL_YFINANCE` 改成 `True` 後執行。若不安裝也沒關係，後面會使用合成資料。

In [ ]:
INSTALL_YFINANCE = False

if INSTALL_YFINANCE:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'yfinance'])
    print('yfinance installed. Please rerun the package check cell.')
else:
    print('Skip installation. This is fine for class because notebooks include a synthetic-data fallback.')

## 3. 建立課程資料

預設使用合成 OHLCV 資料，避免課前因網路、Yahoo Finance 或套件版本卡住。若你想測試真實資料，將 `USE_ONLINE_DATA` 改成 `True`。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.grid'] = True

USE_ONLINE_DATA = False
SYMBOL = '0050.TW'

def make_synthetic_ohlcv(n=760, seed=42):
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range('2021-01-01', periods=n)
    t = np.arange(n)
    regime = np.select([t < n * 0.33, t < n * 0.66], [0.00045, -0.00015], default=0.00025)
    shocks = rng.normal(0, 0.011, n)
    ret = regime + shocks + 0.08 * np.r_[0, shocks[:-1]]
    close = 100 * np.exp(np.cumsum(ret))
    open_ = close * (1 + rng.normal(0, 0.003, n))
    high = np.maximum(open_, close) * (1 + rng.uniform(0.001, 0.012, n))
    low = np.minimum(open_, close) * (1 - rng.uniform(0.001, 0.012, n))
    volume = rng.lognormal(mean=15.2, sigma=0.25, size=n) * (1 + 8 * np.abs(ret))
    df = pd.DataFrame({'Open': open_, 'High': high, 'Low': low, 'Close': close, 'Adj Close': close, 'Volume': volume.astype(int)}, index=dates)
    df.index.name = 'Date'
    return df

def load_market_data(symbol=SYMBOL, start='2020-01-01', use_online=USE_ONLINE_DATA):
    if use_online:
        try:
            import yfinance as yf
            df = yf.download(symbol, start=start, auto_adjust=False, progress=False)
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            if not df.empty and {'Open', 'High', 'Low', 'Close', 'Volume'}.issubset(df.columns):
                print(f'Loaded online data: {symbol}, rows={len(df)}')
                return df.dropna()
        except Exception as exc:
            print('Online download failed; using synthetic data instead.')
            print(type(exc).__name__, exc)
    print('Using synthetic OHLCV data.')
    return make_synthetic_ohlcv()

raw = load_market_data()
raw.head()

## 4. 建立 features 與 target

這裡只做最基本的欄位，讓學生確認後續四份 notebook 的資料處理可以執行。

In [ ]:
def add_features(raw):
    df = raw.copy()
    df['ret_1d'] = df['Close'].pct_change()
    df['ret_fwd_1d'] = df['Close'].shift(-1) / df['Close'] - 1
    df['ma_5'] = df['Close'].rolling(5).mean()
    df['ma_20'] = df['Close'].rolling(20).mean()
    df['ma_gap'] = df['ma_5'] / df['ma_20'] - 1
    df['mom_5'] = df['Close'] / df['Close'].shift(5) - 1
    df['mom_20'] = df['Close'] / df['Close'].shift(20) - 1
    df['vol_20'] = df['ret_1d'].rolling(20).std() * np.sqrt(252)
    df['range_pct'] = (df['High'] - df['Low']) / df['Close']
    df['volume_z'] = (df['Volume'] - df['Volume'].rolling(20).mean()) / df['Volume'].rolling(20).std()
    df = df.dropna()
    df['target_up'] = (df['ret_fwd_1d'] > 0).astype(int)
    return df

features = add_features(raw)
features[['Close', 'ret_1d', 'ma_gap', 'mom_20', 'vol_20', 'volume_z', 'target_up']].tail()

## 5. 快速畫圖檢查

若圖表可以顯示，表示 Colab 圖形環境正常。

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
raw['Close'].plot(ax=axes[0], title='Close price')
features['ret_1d'].plot(ax=axes[1], title='Daily return')
plt.tight_layout()
plt.show()

print('Rows in raw:', len(raw))
print('Rows in features:', len(features))
print('Target up ratio:', round(features['target_up'].mean(), 4))

## 6. 儲存資料包

這會在 Colab 工作目錄建立 `ai_quant_data` 資料夾。正式課堂 notebook 即使不使用這些 CSV，也可以用同樣邏輯自行產生資料。

In [ ]:
from pathlib import Path

out_dir = Path('ai_quant_data')
out_dir.mkdir(exist_ok=True)

raw.to_csv(out_dir / 'market_data.csv')
features.to_csv(out_dir / 'features.csv')
(out_dir / 'README.txt').write_text(
    '智慧金融科技課程資料\n'
    'market_data.csv: OHLCV price data\n'
    'features.csv: engineered features and target\n'
    'For education only. Not investment advice.\n',
    encoding='utf-8'
)

print('Saved files:')
for p in sorted(out_dir.iterdir()):
    print('-', p)

## 7. 課前提交檢查

請在第一堂課前確認：

- 我可以開啟 Colab notebook。
- 我可以執行到最後一格。
- 我知道 `target_up` 是下一期是否上漲，不是今天是否上漲。
- 我知道本課程所有策略與回測只供教育用途，不構成投資建議。
- 若我遇到錯誤，我已截圖並記下錯誤 cell。